# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step approach to loading and exploring the FAIR^2 dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library, referencing all dataset entities by their `@id` fields as defined in the Croissant metadata.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# If not already present, install mlcroissant
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. We'll examine high-level metadata and print the title and description.

In [ ]:
import mlcroissant as mlc
import pandas as pd
from pprint import pprint

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata (the metadata object provides useful attributes directly)
print(f"Dataset Title: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Version: {dataset.metadata.version}")
print(f"License: {dataset.metadata.license}")


## 2. Data Overview
Let's enumerate the available `RecordSet` objects (tables) in this dataset and review their fields and respective `@id` values.

**NOTE:** All entities are referenced by their `@id` as per Croissant best practices.

We display all record sets, then for each, the fields and columns available.

In [ ]:
# List available record sets by @id
record_sets = list(dataset.metadata.record_sets)
print(f"Number of record sets found: {len(record_sets)}\n")

for rs in record_sets:
    print(f"Record Set Name: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Description: {rs.description}")
    # List all fields (columns/variables)
    print("  Fields (@id : name):")
    for field in rs.fields:
        print(f"    - {field.id} : {field.name} (type: {field.data_type})")
    print("")

## 3. Data Extraction
Extract data from a specific record set using its `@id`. As an example, we'll extract and display one of the main data tables for exploratory analysis, referencing everything by `@id`.

*If there are multiple record sets, feel free to adapt the cell below to your use case by choosing the right `@id` values as shown above.*

In [ ]:
# Prepare a dictionary for all tables, indexed by record set @id
dataframes = {}

# Load all record set @ids
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]

# Load records for each record set
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if len(records) > 0:
        dataframes[rs_id] = pd.DataFrame(records)

# Show available DataFrames
print("Available DataFrame keys (record_set @id):")
for key in dataframes.keys():
    print(f" - {key}, columns: {dataframes[key].columns.tolist()}")

# Display first few rows of the primary data table (choose your main record_set @id)
if dataframes:
    main_rs_id = list(dataframes.keys())[0]  # e.g., pick the first non-empty table
    print(f"\nSample data from main record set: {main_rs_id}")
    display(dataframes[main_rs_id].head())
else:
    print("No records found in any record set.")

## 4. Exploratory Data Analysis (EDA)
Let's filter, normalize, and group the data using only entity `@id`s (field names as columns).

- We choose a numeric field's `@id` for filtering and normalization
- Optionally, group by another field using its `@id` (e.g., to view means by group)

⚠️ **Update the variable values below (`numeric_field_id`, `group_field_id`) to match available field `@id`s for your selected record set as found in Section 2/above.**

In [ ]:
# ---- User sets these values to match their dataset's structure ----
# Choose the main data table / record set by its @id
record_set_id = main_rs_id if dataframes else None  # Use first record set found
# Select the numeric field to filter and analyze by its @id, as found in overview above, e.g.:
# numeric_field_id = 'Mass_Spectrometry_Abundance' (replace with the actual field @id from above!)
if record_set_id is not None:
    sample_cols = dataframes[record_set_id].columns.tolist()
    # Try to auto-select a float/integer-looking column
    import re
    auto_numeric_id = next((c for c in sample_cols if re.search(r"(?:abund|count|percent|intensity|weight|score|mw|pi|coverage|peptide|area)", c, re.I)), sample_cols[0])
    numeric_field_id = auto_numeric_id
    print(f"Selected numeric_field_id: {numeric_field_id}")
    
    # Also choose a likely group field (e.g. sample, protein_class, etc)
    auto_group_id = next((c for c in sample_cols if c != numeric_field_id and re.search(r"desc|type|group|cell|class|sample|id|access|protein", c, re.I)), None)
    group_field_id = auto_group_id if auto_group_id else sample_cols[0]
    print(f"Selected group_field_id: {group_field_id}")
else:
    print("No record set available for EDA.")

# ---- Filtering, normalization, grouping ----
if record_set_id is not None:
    df = dataframes[record_set_id]
    # Make sure numeric
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = df[numeric_field_id].quantile(0.75)  # filter: top quartile
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())
    
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    
    # Group by group_field if present
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
        display(grouped_df.head())
else:
    print("Skipped EDA: No appropriate record set dataframe.")

## 5. Visualization
Let's quickly plot a histogram and boxplot of the selected numeric field, and if grouping exists, a bar plot of the group means. Feel free to adjust the field and group names as appropriate for your data.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize main numeric field distribution
if record_set_id is not None:
    plt.figure(figsize=(10,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    # Boxplot by group, if reasonable field exists
    if group_field_id in df.columns:
        plt.figure(figsize=(12,5))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Visualization skipped: no suitable record set/")

## 6. Conclusion

- We loaded the FAIR^2 dataset with `mlcroissant`, referencing all entities by their `@id` fields, and explored its record sets and main numeric properties.
- Example EDA and visualizations demonstrated how to filter, normalize, and group data by their fields for further biological or quantitative analysis.
- For advanced work (e.g., training models), repeat and adapt these steps with additional fields and record sets.

> **Next steps:** Try linking fields with domain information in the Croissant metadata, reuse group/numeric field `@id`s, and consult the metadata inspector for new record sets or further data integration.